In [1]:
import sys
import platform

print(sys.version)
print(platform.platform())
!nvidia-smi

3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Linux-6.6.159+-x86_64-with-glibc2.39
Thu Nov 12 21:27:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   33C    P0             57W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |         

In [2]:
!git clone --recurse-submodules https://github.com/rarevalo13/MusicLLM.git /content/MusicLLM

Cloning into '/content/MusicLLM'...
remote: Enumerating objects: 27, done.
remote: Counting objects: 100% (27/27), done.
remote: Compressing objects: 100% (23/23), done.
remote: Total 27 (delta 4), reused 25 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (27/27), 96.49 KiB | 12.06 MiB/s, done.
Resolving deltas: 100% (4/4), done.
Submodule 'stable-audio-3' (https://github.com/Stability-AI/stable-audio-3) registered for path 'stable-audio-3'
Cloning into '/content/MusicLLM/stable-audio-3'...
remote: Enumerating objects: 2667, done.        
remote: Counting objects: 100% (922/922), done.        
remote: Compressing objects: 100% (264/264), done.        
remote: Total 2667 (delta 770), reused 671 (delta 658), pack-reused 1745 (from 1)        
Receiving objects: 100% (2667/2667), 31.90 MiB | 15.16 MiB/s, done.
Resolving deltas: 100% (1529/1529), done.
Submodule path 'stable-audio-3': checked out '77e0398c26f7d22d801ec1bb036307e8b231f741'


In [3]:
%cd /content/MusicLLM/stable-audio-3
%pip install --only-binary=torch,torchaudio -e ".[lora]"

/content/MusicLLM/stable-audio-3
Obtaining file:///content/MusicLLM/stable-audio-3
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.4/832.4 kB 41.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 30.7 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 126.9 MB/s  0:00:020:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 193.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 215.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 51.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 36.5 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 162.2 MB/s  0:00:010:00:01

In [2]:
%pip install --only-binary=:all: "torch==2.7.1" "torchaudio==2.7.1" "torchvision==0.22.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 117.4 MB/s  0:00:00
  Attempting uninstall: torchvision
    Found existing installation: torchvision 0.29.1+cu130
    Uninstalling torchvision-0.29.1+cu130:
      Successfully uninstalled torchvision-0.29.1+cu130


In [2]:
import torch
import torchvision
import torchaudio
import pytorch_lightning as pl

print("Torch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("Torchaudio:", torchaudio.__version__)
print("Lightning:", pl.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Torch: 2.7.1+cu126
Torchvision: 0.22.1+cu126
Torchaudio: 2.7.1+cu126
Lightning: 2.5.5
CUDA available: True
GPU: NVIDIA A100-SXM4-80GB


In [3]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


In [4]:
!tar -xzf /content/rock_lora_smoke.tar.gz -C /content/MusicLLM

tar (child): /content/rock_lora_smoke.tar.gz: Cannot open: No such file or directory
tar (child): Error is not recoverable: exiting now
tar: Child returned status 2
tar: Error is not recoverable: exiting now


In [ ]:
from pathlib import Path

root = Path("/content/MusicLLM/rock_lora_smoke")
clips = sorted(root.glob("*.wav"))
assert clips, "No WAV files found"
assert all(p.with_suffix(".txt").is_file() for p in clips)
print(f"{len(clips)} clips with matching captions")

# 100 Step Smoke Test on Rock Genre

In [10]:
from pathlib import Path

drive = Path("/content/drive/MyDrive")
print("MyDrive exists:", drive.is_dir())

if drive.is_dir():
    print("Top-level files:")
    for p in sorted(drive.iterdir()):
        print(p.name)
else:
    print("Mount Drive first using Colab: Mount Google Drive to Server…")

MyDrive exists: False
Mount Drive first using Colab: Mount Google Drive to Server…


In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
from pathlib import Path

archive = Path("/content/drive/MyDrive/rock_lora_smoke.tar.gz")
print("Archive found:", archive.is_file())

Archive found: True


In [14]:
!tar -xzf /content/drive/MyDrive/rock_lora_smoke.tar.gz -C /content/MusicLLM
!ls -lh /content/MusicLLM/rock_lora_smoke

tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.xattr.com.apple.provenance'
tar: Ignoring unknown extended header keyword 'LIBARCH

In [15]:
%cd /content/MusicLLM/stable-audio-3

!python scripts/train_lora.py \
    --model small-music-base \
    --data_dir ../rock_lora_smoke \
    --rank 16 \
    --adapter_type dora-rows \
    --exclude seconds_total \
    --duration 30 \
    --batch_size 1 \
    --steps 100 \
    --checkpoint_every 100 \
    --demo_every 100 \
    --save_dir /content/lora_rock_smoke

/content/MusicLLM/stable-audio-3
No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention
No module named 'flash_attn'
flash_attn varlen/bert_padding not available, disabling varlen attention
Seed set to 42
/usr/local/lib/python3.13/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)
Found 20 files
LoRA config: rank=16, alpha=16, adapter_type=dora-rows
  exclude: ['seconds_total']
lora layers: 192
Demo sample 0: prompt=Instrumental rock with experimental influences, improvisation, contemporary mood, medium tempo seconds_total=30
Demo sample 1: prompt=Instrumental rock with experimental influences, improvisation, contemporary mood, medium tempo seconds_total=30
Demo sample 2: prompt=Instrumental rock with pop and ambient influences, peaceful, soft, upbeat mood, medium tempo seconds_total=30
Demo sample 3: prompt=I

In [18]:
from pathlib import Path

for root in [
    Path("/content/lora_rock_smoke"),
    Path("/content/drive/MyDrive/MusicLLM_runs/lora_rock_smoke"),
    Path("/content/MusicLLM/stable-audio-3"),
]:
    print(f"\nChecking {root} (exists: {root.exists()})")
    for p in root.rglob("*.ckpt"):
        print(p, f"({p.stat().st_size / 1024**2:.1f} MB)")


Checking /content/lora_rock_smoke (exists: True)
/content/lora_rock_smoke/epoch=4-step=100.ckpt (41.0 MB)

Checking /content/drive/MyDrive/MusicLLM_runs/lora_rock_smoke (exists: False)

Checking /content/MusicLLM/stable-audio-3 (exists: True)


In [19]:
from pathlib import Path
import shutil

assert Path("/content/drive/MyDrive").is_dir(), "Mount Google Drive first"

backup = Path("/content/drive/MyDrive/MusicLLM_runs/lora_rock_smoke")
shutil.copytree("/content/lora_rock_smoke", backup, dirs_exist_ok=True)

checkpoint = backup / "epoch=4-step=100.ckpt"
assert checkpoint.is_file()
print("Backed up:", checkpoint)

Backed up: /content/drive/MyDrive/MusicLLM_runs/lora_rock_smoke/epoch=4-step=100.ckpt


In [20]:
%cd /content/MusicLLM/stable-audio-3
!python ../compare_lora.py --help

/content/MusicLLM/stable-audio-3
No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention
No module named 'flash_attn'
flash_attn varlen/bert_padding not available, disabling varlen attention
usage: compare_lora.py [-h] --lora LORA [--model MODEL] [--duration DURATION]
                       [--out OUT]

options:
  -h, --help           show this help message and exit
  --lora LORA
  --model MODEL
  --duration DURATION
  --out OUT


In [21]:
%cd /content/MusicLLM/stable-audio-3

!python ../compare_lora.py \
    --lora /content/lora_rock_smoke/epoch=4-step=100.ckpt \
    --duration 30 \
    --out /content/drive/MyDrive/MusicLLM_runs/lora_rock_smoke/comparison

/content/MusicLLM/stable-audio-3
No module named 'flash_attn'
flash_attn not installed, disabling Flash Attention
No module named 'flash_attn'
flash_attn varlen/bert_padding not available, disabling varlen attention
model_config.json: 100% 10.3k/10.3k [00:00<00:00, 4.88MB/s]

model.safetensors: downloading bytes:   0% 0.00/2.27G [00:00<?, ?B/s]s]
model.safetensors: downloading bytes:  33% 751M/2.27G [00:02<00:02, 638MB/s, 58.5MB/s  ]  
model.safetensors: downloading bytes:  50% 1.13G/2.27G [00:03<00:01, 761MB/s, 89.9MB/s  ]]
model.safetensors: downloading bytes:  54% 1.22G/2.27G [00:03<00:01, 786MB/s, 99.5MB/s  ]]
model.safetensors: downloading bytes:  61% 1.38G/2.27G [00:03<00:01, 778MB/s,  111MB/s  ] ]
model.safetensors: downloading bytes:  75% 1.71G/2.27G [00:03<00:00, 937MB/s,  136MB/s  ] ]
model.safetensors: downloading bytes:  81% 1.85G/2.27G [00:03<00:00, 819MB/s,  150MB/s  ] ]
model.safetensors: downloading bytes:  86% 1.95G/2.27G [00:04<00:00, 706MB/s,  159MB/s  ] ]
model.safe

In [22]:
from pathlib import Path

checkpoint = Path(
    "/content/drive/MyDrive/MusicLLM_runs/"
    "lora_rock_smoke/epoch=4-step=100.ckpt"
)
assert checkpoint.is_file(), "Checkpoint is not backed up!"
print(checkpoint, checkpoint.stat().st_size / 1024**2, "MB")

/content/drive/MyDrive/MusicLLM_runs/lora_rock_smoke/epoch=4-step=100.ckpt 41.010008811950684 MB


In [23]:
from google.colab import runtime
runtime.unassign()